# VOICE_CLONE_TTS — Nhân bản giọng nói (Text-to-Speech) — OmniVoice (600+ ngôn ngữ)
## Cách dùng nhanh: bấm "Chạy tất cả" (Run all), đợi cài đặt + tải model là dùng được.

VUI LÒNG ĐỌC HẾT HƯỚNG DẪN BÊN DƯỚI TRƯỚC KHI SỬ DỤNG.

---

## GIỚI THIỆU
* Bản này dùng model **OmniVoice** (https://github.com/k2-fsa/OmniVoice) — model TTS nguồn mở (giấy phép **Apache-2.0**, dùng được cả cho thương mại), hỗ trợ **600+ ngôn ngữ trong MỘT model**, trong đó có tiếng Việt. Rất nhanh (nhanh hơn gấp nhiều lần realtime) và ngắt nghỉ theo dấu câu tự nhiên.
* Chức năng: **nhân bản giọng nói** — tải lên 1 file giọng mẫu (3–10 giây), nhập văn bản bất kỳ, máy sẽ đọc bằng đúng giọng đó. Không cần chọn model theo từng ngôn ngữ như bản F5-TTS nữa.
* **Chế độ không cần giọng mẫu**: điền ô "Mô tả giọng" (VD: `female, low pitch, british accent`) là model tự dựng giọng theo mô tả; hoặc để trống cả hai thì model tự chọn giọng (auto).
* Hỗ trợ **KHÔNG GIỚI HẠN KÍ TỰ** — văn bản dài được tự chia đoạn ~15 giây và ghép lại mượt mà, VRAM gần như không đổi.
* Có thể điều khiển cảm xúc/ngắt giọng bằng tag nội dòng: `[laughter]`, `[question-en]`, `[surprise-ah]`...

## 👉 HƯỚNG DẪN:
1. Trên trình đơn Colab: **Runtime -> Change runtime type -> T4 GPU** (cực kì quan trọng).
2. **Bấm "Chạy tất cả"** để cài đặt và khởi động.
3. Lần chạy đầu tiên sẽ tải model OmniVoice (~7 GB) và mất vài phút.
4. Sau khi có link giao diện (dòng cuối cùng của TASK 8), mở trong tab mới.
5. Tải lên **file giọng mẫu** (WAV/MP3 sạch, không nhạc nền, 3–10 giây), nhập văn bản, bấm **Synthesize**.
6. Chọn **Ngôn ngữ** cho kết quả (VD: Tiếng Việt, English...) — chọn đúng ngôn ngữ giúp chuẩn hơn; để "Tự động" model tự đoán.
7. **Lưu giọng để dùng lại**: điền Tên giọng rồi bấm `Lưu giọng mẫu này` — lần sau chỉ cần chọn tên trong danh sách `Giọng đã lưu`.
8. (Tùy chọn) Chạy cell **TASK 6 — GẮN GOOGLE DRIVE** để giọng đã lưu được giữ lâu dài giữa các phiên Colab.

## NẾU KẾT QUẢ CHƯA ƯNG:
* **Đọc thiếu chữ / lướt nhanh**: tăng **num_step** lên 48–64, hoặc tăng nhẹ **guidance** lên 2.5–3.0.
* **Đọc chậm / kéo dài**: tăng **Tốc độ đọc** (speed) lên 1.1–1.3.
* **Giọng mẫu quá dài (>20s)**: model sẽ chậm hơn và có thể kém giống — nên cắt còn 3–10 giây.
* Nhập chính xác **"Nội dung trong file giọng mẫu"** hoặc để trống + bật checkbox tự nhận dạng (Whisper) để máy khớp đúng độ dài.

## ⚠️ LƯU Ý:
* Chỉ dùng cho mục đích cá nhân / học tập. **Không** nhân bản giọng nói của người khác khi chưa được phép.
* Model OmniVoice dùng giấy phép **Apache-2.0** (miễn phí, cho phép dùng thương mại).


In [ ]:
# @title TASK 1 — KIỂM TRA MÔI TRƯỜNG (GPU / FFMPEG)
import subprocess, sys

# 1) Kiểm tra GPU — BẮT BUỘC cần GPU; không có GPU thì gen rất chậm (mỗi câu mất nhiều phút)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM: {free/2**30:.1f} GB free / {total/2**30:.1f} GB")
else:
    raise SystemExit("❌ KHÔNG CÓ GPU. Vào Runtime -> Change runtime type -> chọn T4 GPU, rồi chạy lại từ đầu. (Không có GPU thì TTS gần như không dùng được — cực chậm.)")

# 2) Cài ffmpeg nếu chưa có (cần để đọc/xuất âm thanh)
try:
    subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    print("ffmpeg: OK")
except Exception:
    print("ffmpeg: đang cài đặt...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)
    print("ffmpeg: xong")


In [ ]:
# @title TASK 2 — CÀI ĐẶT OmniVoice
# Trên Colab, torch/torchaudio đã có sẵn. pip install omnivoice sẽ bổ sung các thư viện còn thiếu.
import subprocess, sys
try:
    import omnivoice  # noqa: F401
    print("✅ OmniVoice đã được cài sẵn — bỏ qua cài đặt, sang bước tiếp theo.")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice"], check=True)
    print("Đã cài xong OmniVoice.")

# ⚡ TĂNG TỐC: cài FlashInfer (nhanh hơn 2-2.9x, KHÔNG mất chất lượng) — chỉ chạy khi có GPU.
# Cài lỗi thì tự bỏ qua (vẫn gen được, chỉ chậm hơn) — không làm hỏng notebook.
def _try_install_flashinfer():
    import importlib.util, subprocess, sys
    if importlib.util.find_spec("flashinfer"):
        return True
    ok = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                         "flashinfer-python==0.6.15.post1",
                         "flashinfer-jit-cache==0.6.15.post1+cu128",
                         "--extra-index-url", "https://flashinfer.ai/whl/cu128/"],
                        check=False).returncode == 0
    if not ok:
        ok = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flashinfer-python"],
                            check=False).returncode == 0
    return ok

try:
    import torch
    _flash_ok = _try_install_flashinfer() if torch.cuda.is_available() else False
    print("FlashInfer:", "OK — gen nhanh hơn 2-2.9x." if _flash_ok
          else "bỏ qua (không có GPU NVIDIA hoặc cài lỗi) — vẫn chạy được, chỉ chậm hơn.")
except Exception as e:
    print("⚠️ FlashInfer không cài được (không sao, vẫn chạy):", e)


In [ ]:
# @title TASK 3 — TẢI & NẠP MODEL OmniVoice (600+ ngôn ngữ, 1 model duy nhất)
import os
import torch
from omnivoice import OmniVoice

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Danh sách ngôn ngữ (chọn đúng giúp model chuẩn hơn; để 'Tự động' model tự đoán).
# Chỉ dùng để hiển thị + lưu giọng theo từng ngôn ngữ. Model thì CHỈ CÓ MỘT.
LANGUAGES = {
    "🌐 Tự động (model tự đoán)": None,
    "🇻🇳 Vietnamese": "Vietnamese",
    "🇬🇧 English": "English",
    "🇨🇳 Chinese": "Chinese",
    "🇯🇵 Japanese": "Japanese",
    "🇰🇷 Korean": "Korean",
    "🇫🇷 French": "French",
    "🇩🇪 German": "German",
    "🇮🇹 Italian": "Italian",
    "🇪🇸 Spanish": "Spanish",
    "🇷🇺 Russian": "Russian",
    "🇮🇳 Hindi": "Hindi",
    "🇹🇭 Thai": "Thai",
    "🇮🇩 Indonesian": "Indonesian",
    "🇵🇹 Portuguese": "Portuguese",
    "🇳🇱 Dutch": "Dutch",
    "🇹🇷 Turkish": "Turkish",
    "🇸🇦 Arabic": "Arabic",
}

engine = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=DEVICE,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
print("Sampling rate:", engine.sampling_rate)

# ⚡ Kích hoạt FlashInfer nếu đã cài (nhanh ~2.4x ngay cả với 1 câu; lỗi thì tự bỏ qua).
try:
    from omnivoice.models.omnivoice_flashinfer import apply_flashinfer
    apply_flashinfer(engine, enable_cuda_graph=True)
    print("FlashInfer: đã kích hoạt (CUDA graph) — gen nhanh hơn.")
except Exception:
    print("FlashInfer: không dùng được — gen bằng kernel mặc định (vẫn chạy, chỉ chậm hơn).")

print("OK — model đã sẵn sàng (1 model dùng được cho 600+ ngôn ngữ).")


In [ ]:
# @title TASK 4 — HÀM SINH ỔN ĐỊNH (chống thiếu chữ / vấp, dùng chung cho các task sau)
# OmniVoice tự ngắt nghỉ theo dấu câu, tự chia văn bản dài thành từng đoạn ~15 giây
# rồi ghép lại mượt (cross-fade) — nên không cần phải tự chia nhỏ text như bản F5-TTS.
import os, random, sys
import numpy as np
import torch
import soundfile as sf

def omnivoice_generate(engine, gen_text, ref_audio=None, ref_text=None,
                       language=None, instruct=None, speed=1.0,
                       num_step=16, guidance=2.0, seed=None, **extra):
    """Sinh giọng bằng OmniVoice. Trả về (wave, sr, seed_da_dung).
    - ref_audio + ref_text  = nhân bản giọng (voice cloning)
    - instruct              = thiết kế giọng theo mô tả (không cần giọng mẫu)
    - không có gì           = auto voice (model tự chọn giọng)"""
    if seed is None:
        seed = random.randint(0, 2**32 - 1)
    else:
        seed = int(seed) & 0xFFFFFFFF
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    kwargs = dict(
        text=gen_text,
        speed=float(speed),
        num_step=int(num_step),
        guidance_scale=float(guidance),
    )
    if language:
        kwargs["language"] = language
    if ref_audio:
        kwargs["ref_audio"] = ref_audio
    if ref_text:
        kwargs["ref_text"] = ref_text
    if instruct:
        kwargs["instruct"] = instruct
    kwargs.update(extra)  # audio_chunk_duration, postprocess_output, ...

    audios = engine.generate(**kwargs)
    wave = audios[0]
    if hasattr(wave, "detach"):
        wave = wave.detach().cpu()
    wave = np.asarray(wave, dtype=np.float32)
    if wave.ndim > 1:
        wave = wave.squeeze()
    return wave, engine.sampling_rate, seed


def save_wav(wave, sr, path):
    wave = np.asarray(wave)
    if wave.ndim > 1:
        wave = wave.squeeze()
    sf.write(path, wave, sr)
    return path

def to_mp3(path, bitrate=128):
    """Nén WAV -> MP3 để file nhỏ, phát/tua mượt trên Colab."""
    mp3_path = os.path.splitext(path)[0] + ".mp3"
    try:
        os.system(f'ffmpeg -y -loglevel error -i "{path}" -b:a {bitrate}k -write_xing 1 "{mp3_path}"')
        if os.path.exists(mp3_path) and os.path.getsize(mp3_path) > 0:
            return mp3_path
    except Exception:
        pass
    return path


In [ ]:
# @title TASK 5 — CHẠY THỬ (tự tạo 1 giọng mẫu tạm rồi nhân bản)
import os
os.makedirs("/content/output", exist_ok=True)

# Bước 1: tạo 1 giọng mẫu tạm bằng chế độ auto voice (model tự chọn giọng)
ref_audio = "/content/output/ref_voice.wav"
ref_text = "Some call me nature, others call me mother nature."
wave, sr, _ = omnivoice_generate(engine, ref_text, language="English", num_step=16, seed=123)
save_wav(wave, sr, ref_audio)
print("Đã tạo giọng mẫu tạm:", ref_audio)

# Bước 2: nhân bản đúng giọng đó để đọc đoạn văn cần test
gen_text = "Hello! This is a quick test of your cloned voice. " \
           "The model pauses naturally at commas and full stops, " \
           "and it reads every single word fully and clearly."
wave, sr, seed = omnivoice_generate(engine, gen_text, ref_audio=ref_audio, ref_text=ref_text,
                                    language="English", num_step=16, seed=456)
out_path = save_wav(wave, sr, "/content/output/test_output.wav")
print("Đã tạo:", out_path, "| seed =", seed, "| sr =", sr)

from IPython.display import Audio
Audio(to_mp3(out_path))


In [ ]:
# @title TASK 6 — GẮN GOOGLE DRIVE (tùy chọn, để giọng lưu được lâu dài)
# KHÔNG bắt buộc. Chạy cell này nếu muốn các giọng đã lưu được giữ lại giữa các phiên Colab.
# (Không gắn Drive thì giọng vẫn lưu được trong phiên hiện tại, nhưng mất khi đóng runtime.)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Đã gắn Google Drive. Giọng đã lưu sẽ được lưu vào MyDrive/VOICE_CLONE_TTS/voices/")
except Exception as e:
    print("Bỏ qua gắn Drive (không sao):", e)


In [ ]:
# @title TASK 7 — THƯ VIỆN GIỌNG ĐÃ LƯU (lưu / nạp lại giọng mẫu đã dùng)
# Lưu nhiều giọng mẫu đã dùng (kèm nội dung) để dùng lại nhanh. Nếu đã gắn Google Drive
# (cell TASK 6) giọng sẽ được giữ lâu dài; nếu không, giọng lưu trong phiên hiện tại.
import os, json, shutil, re

VOICE_DIR = "/content/voices"
DRIVE_VOICE_DIR = "/content/drive/MyDrive/VOICE_CLONE_TTS/voices"

def _voices_root():
    if os.path.isdir("/content/drive/MyDrive"):
        return DRIVE_VOICE_DIR
    return VOICE_DIR

def _load_index():
    index_path = os.path.join(_voices_root(), "voices.json")
    if os.path.exists(index_path):
        with open(index_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def _save_index(index):
    root = _voices_root()
    os.makedirs(root, exist_ok=True)
    with open(os.path.join(root, "voices.json"), "w", encoding="utf-8") as f:
        json.dump(index, f, ensure_ascii=False, indent=2)

def _safe_name(name):
    name = re.sub(r"[^\w\- ]", "", name).strip()
    return name or "voice"

AUDIO_EXTS = (".wav", ".mp3", ".flac", ".m4a", ".ogg", ".wma")

def _model_folder(model):
    """Tên thư mục riêng cho mỗi ngôn ngữ (mỗi ngôn ngữ = 1 thư mục giọng)."""
    if not model:
        return "Chung"
    f = re.sub(r"[^\w]+", "_", model).strip("_")
    f = re.sub(r"_+", "_", f)
    return f or "Chung"

def _model_dir(model):
    return os.path.join(_voices_root(), _model_folder(model))

def _model_from_folder(folder):
    """Ngược lại: tên thư mục -> ngôn ngữ (key trong LANGUAGES)."""
    if "LANGUAGES" in globals():
        for mk in LANGUAGES:
            if _model_folder(mk) == folder:
                return mk
    return folder

def list_voices():
    return list_voices_by_model(None)

def list_voices_by_model(model=None):
    """Liệt kê giọng của 1 ngôn ngữ bằng cách QUÉT thư mục ngôn ngữ đó.
    Bạn có thể bỏ thẳng file WAV/MP3 vào thư mục giọng của ngôn ngữ là nó tự hiện ra."""
    index = _load_index()
    names = set()
    if model is not None:
        folder = _model_dir(model)
        if os.path.isdir(folder):
            for fn in os.listdir(folder):
                if fn.lower().endswith(AUDIO_EXTS):
                    names.add(fn)
        for v in index.values():
            if (not v.get("model") or v["model"] == model) and v.get("audio"):
                names.add(os.path.basename(v["audio"]))
    else:
        for _, _, files in os.walk(_voices_root()):
            for fn in files:
                if fn.lower().endswith(AUDIO_EXTS):
                    names.add(fn)
        for v in index.values():
            if v.get("audio"):
                names.add(os.path.basename(v["audio"]))
    return sorted(names, key=str.lower)

def save_voice(name, audio_path, ref_text, model=None):
    """Lưu giọng vào THƯ MỤC của ngôn ngữ đang chọn."""
    if not (name or "").strip():
        return list_voices_by_model(model), "⚠️ Chưa nhập Tên giọng để lưu."
    if not audio_path:
        return list_voices_by_model(model), "⚠️ Chưa có file giọng mẫu (ref audio) để lưu."
    name = _safe_name(name)
    ext = os.path.splitext(audio_path)[1] or ".wav"
    folder = _model_folder(model)
    dest_dir = os.path.join(_voices_root(), folder)
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, name + ext)
    for fn in os.listdir(dest_dir):  # xoa cac file cung ten khac duoi (.wav/.mp3) de khong trung
        if fn.lower().endswith(AUDIO_EXTS) and os.path.splitext(fn)[0] == name and os.path.abspath(os.path.join(dest_dir, fn)) != os.path.abspath(dest):
            try: os.remove(os.path.join(dest_dir, fn))
            except Exception: pass
    shutil.copyfile(audio_path, dest)
    index = _load_index()
    index[name] = {"audio": dest, "ref_text": ref_text or "", "model": model or ""}
    _save_index(index)
    return list_voices_by_model(model), f"Đã lưu giọng {name} vào thư mục {folder}/ — chọn lại là dùng được."

def load_voice(name):
    """Nạp giọng -> (path, ref_text, model/ngôn ngữ). Hỗ trợ tên có/khoảng đuôi .wav/.mp3."""
    if not name:
        return None, None, ""
    index = _load_index()
    v = index.get(name)
    if v is None:
        for k, it in index.items():
            if os.path.basename(it["audio"]) == name:
                v = it
                break
    if v and v.get("audio") and os.path.exists(v["audio"]):
        return v["audio"], v.get("ref_text", ""), v.get("model", "")
    path, ref_text, model = None, "", ""
    for root_, _, files in os.walk(_voices_root()):
        for fn in files:
            if fn.lower().endswith(AUDIO_EXTS) and (fn == name or os.path.splitext(fn)[0] == name):
                path = os.path.join(root_, fn)
                break
        if path:
            break
    if path:
        folder = os.path.basename(os.path.dirname(path))
        model = _model_from_folder(folder)
        for k, it in index.items():
            if os.path.basename(it["audio"]) == os.path.basename(path):
                ref_text = it.get("ref_text", "")
                model = it.get("model", "") or model
                break
        return path, ref_text, model
    return None, None, ""

def delete_voice(name):
    """Xóa giọng (theo index hoặc file trực tiếp trong thư mục)."""
    index = _load_index()
    if name and name in index:
        audio = index[name].get("audio")
        del index[name]
        _save_index(index)
        if audio and os.path.exists(audio):
            os.remove(audio)
        return list_voices(), f"Đã xóa giọng {name}."
    for root_, _, files in os.walk(_voices_root()):
        for fn in files:
            if fn == name or os.path.splitext(fn)[0] == name:
                fp = os.path.join(root_, fn)
                if os.path.exists(fp):
                    os.remove(fp)
                return list_voices(), f"Đã xóa giọng {name}."
    return list_voices(), "Không tìm thấy giọng đã chọn."

def _voice_is_saved(audio):
    """Kiểm tra file audio đã được lưu trong thư viện hay chưa."""
    if not audio:
        return False
    for v in _load_index().values():
        saved = v.get("audio")
        if saved and os.path.abspath(saved) == os.path.abspath(audio):
            return True
    return False

def _ensure_voice_dirs():
    """Tự tạo thư mục giọng cho từng ngôn ngữ khi chạy lần đầu."""
    root = _voices_root()
    os.makedirs(root, exist_ok=True)
    made = []
    if "LANGUAGES" in globals():
        for mk in LANGUAGES:
            d = _model_dir(mk)
            os.makedirs(d, exist_ok=True)
            made.append((_model_folder(mk), d))
    return root, made

root, made = _ensure_voice_dirs()
print("Thư mục chứa giọng:", root)
print("Đã tự tạo thư mục giọng cho từng ngôn ngữ (lần đầu):")
for folder, path in made:
    print("  ", folder, "->", path)
print("Số giọng đã lưu hiện có:", len(list_voices()))


In [ ]:
# @title TASK 7.5 — TỰ NHẬN DẠNG NỘI DUNG GIỌNG MẪU (Whisper, tùy chọn)
# Khi tải lên/nạp giọng mẫu, tự chuyển giọng nói trong file thành chữ rồi điền vào ô
# 'Nội dung trong file giọng mẫu' (ref_text) để model khớp đúng độ dài.
# Lần đầu sử dụng sẽ tự cài faster-whisper và tải model "small" (~460MB) cho lần nhận dạng đầu.
import os, sys, subprocess

_ASR = None

def _get_asr():
    global _ASR
    if _ASR is not None:
        return _ASR
    try:
        import faster_whisper  # noqa: F401
    except Exception:
        print("Đang cài faster-whisper (chỉ cài 1 lần)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=True)
    import torch
    use_cuda = torch.cuda.is_available()
    from faster_whisper import WhisperModel
    print("Đang nạp model nhận dạng giọng nói (Whisper small)...")
    _ASR = WhisperModel("small", device="cuda" if use_cuda else "cpu",
                       compute_type="float16" if use_cuda else "int8")
    return _ASR

def transcribe_ref(audio, enabled=True):
    """Nhận dạng nội dung file giọng mẫu -> trả về chữ ('' nếu không bật/lỗi/trống)."""
    if not enabled or not audio or not os.path.exists(audio):
        return ""
    try:
        segments, _info = _get_asr().transcribe(audio, beam_size=5)
        text = " ".join(s.text.strip() for s in segments).strip()
        return text or ""
    except Exception as e:
        print("⚠️ Không tự nhận dạng được nội dung giọng mẫu:", e)
        return ""

print("OK — sẵn sàng chức năng tự nhận dạng nội dung giọng mẫu (TASK 7.5).")


In [ ]:
# @title TASK 8 — KHỞI ĐỘNG GIAO DIỆN NHÂN BẢN GIỌNG NÓI
import os, traceback
import gradio as gr

os.makedirs("/content/output", exist_ok=True)

def run_synthesis(lang_key, tasks, speed, num_step, guidance, chunk_duration, postprocess):
    """Gen TUẦN TỰ một danh sách task (dùng chung cho nút Gen hàng chờ).
    Mỗi task = dict(ref_audio, ref_text, instruct, text, seed). Trả về (files, msg).
    Task lỗi sẽ được bỏ qua nhưng vẫn ghi rõ lỗi vào thông báo để bạn biết lý do."""
    tasks = [t for t in (tasks or []) if t and (t.get("text") or "").strip()]
    if not tasks:
        raise gr.Error("Không có văn bản nào để gen — nhập văn bản rồi bấm 'Thêm vào hàng chờ', hoặc gõ thẳng text.")
    lang = LANGUAGES.get(lang_key) if lang_key in LANGUAGES else None
    out_dir = "/content/output"
    os.makedirs(out_dir, exist_ok=True)
    files, done, errors = [], 0, []
    for i, task in enumerate(tasks, start=1):
        try:
            wave, sr, seed_used = omnivoice_generate(
                engine, task["text"],
                ref_audio=task.get("ref_audio"), ref_text=(task.get("ref_text") or "").strip() or None,
                language=lang, instruct=task.get("instruct") or None,
                speed=speed, num_step=num_step, guidance=guidance,
                audio_chunk_duration=float(chunk_duration),
                postprocess_output=bool(postprocess),
                seed=task.get("seed"),
            )
        except Exception as e:
            traceback.print_exc()
            errors.append(f"Task {i}: {type(e).__name__}: {e}")
            continue
        if wave is None:
            errors.append(f"Task {i}: model không sinh được âm thanh.")
            continue
        out = save_wav(wave, sr, os.path.join(out_dir, f"cloned_output_{i:03d}.wav"))
        files.append(to_mp3(out))
        done += 1
    msg = f"✅ Đã gen xong {done}/{len(tasks)} task (tuần tự)."
    if errors:
        msg += "\n⚠️ Lỗi:\n" + "\n".join(errors)
    return files, msg

def gen_queue(lang_key, ref_audio, ref_text, gen_text, instruct, speed, num_step, guidance, chunk_duration, postprocess, seed_text, queue):
    """Gen tuần tự: text hiện tại + toàn bộ hàng chờ bằng 1 lần bấm."""
    if not (gen_text or "").strip() and not (queue or []):
        raise gr.Error("Chưa có gì để gen — nhập văn bản ở ô 'Văn bản cần đọc' hoặc thêm task vào hàng chờ.")
    seed = None
    if (seed_text or "").strip():
        try:
            seed = int(seed_text.strip())
        except ValueError:
            raise gr.Error("Seed phải là số nguyên (để trống nếu muốn ngẫu nhiên).")
    tasks = []
    if (gen_text or "").strip():
        tasks.append({"ref_audio": ref_audio, "ref_text": (ref_text or "").strip(), "instruct": (instruct or "").strip() or None, "text": gen_text, "seed": seed})
    for t in (queue or []):
        tasks.append(t)
    files, msg = run_synthesis(lang_key, tasks, speed, num_step, guidance, chunk_duration, postprocess)
    preview = _dropdown_update([(os.path.basename(f), f) for f in files]) if files else _dropdown_update([])
    first = files[0] if files else None
    return first, msg, files, preview

def _dropdown_update(choices, value=None):
    """Cập nhật Dropdown, tương thích nhiều phiên bản Gradio."""
    try:
        return gr.update(choices=choices, value=value)
    except (AttributeError, TypeError):
        return gr.Dropdown(choices=choices, value=value)

def load_voice_to_ui(name, current_lang):
    """Chọn giọng đã lưu => tự nạp giọng, và cho NGHE thử (bản MP3 nhẹ, tua nhanh)."""
    audio, text, lang = load_voice(name)
    if audio:
        target_lang = lang if (lang and lang in LANGUAGES) else current_lang
        preview = make_preview(audio)
        return preview, text or "", "Nạp giọng " + (name or "") + " xong. Bấm nút play trên ô Giọng mẫu.", preview, target_lang
    return None, "", "Chưa có giọng nào để nạp.", None, current_lang

def refresh_voices():
    return _dropdown_update(list_voices())

PREVIEW_DIR = "/content/preview"  # thu muc tam cho file nghe thu, KHONG nam trong thu vien giọng
def make_preview(audio):
    """Tạo bản MP3 nhẹ (header Xing) trong thu muc tam để nghe/tua nhanh."""
    if not audio:
        return None
    os.makedirs(PREVIEW_DIR, exist_ok=True)
    base = os.path.splitext(os.path.basename(audio))[0]
    mp3 = os.path.join(PREVIEW_DIR, base + "_preview.mp3")
    try:
        os.system(f'ffmpeg -y -loglevel error -i "{audio}" -b:a 128k -write_xing 1 "{mp3}"')
        if os.path.exists(mp3) and os.path.getsize(mp3) > 0:
            return mp3
    except Exception:
        pass
    return audio

def _html_audio(path):
    """Player âm thanh gọn (chỉ nút play, không thanh waveform rộng)."""
    if not path or not os.path.exists(path):
        return "<span style='color:#666;font-size:13px'>Chọn hoặc nạp một giọng để nghe thử.</span>"
    return ("<audio controls preload='metadata' style='width:100%;max-width:340px;height:38px;margin-top:4px'>"
            f"<source src='/file={path}' type='audio/mpeg'></audio>")

def update_preview(audio):
    return _html_audio(make_preview(audio))

def read_txt(file):
    """Đọc nội dung file .txt và điền vào ô 'Văn bản cần đọc'."""
    if not file:
        return ""
    try:
        with open(file, "r", encoding="utf-8", errors="replace") as f:
            content = f.read()
        return content if content.strip() else ""
    except Exception as e:
        raise gr.Error("Không đọc được file txt: " + str(e))

def refresh_voices_by_model(model, current_value=None):
    """Lọc danh sách giọng theo ngôn ngữ đang chọn, giữ nguyên lựa chọn nếu còn tồn tại."""
    choices = list_voices_by_model(model)
    if current_value not in choices:
        current_value = None
    return _dropdown_update(choices, value=current_value)

def load_queue_preview(name):
    """Trả về file audio để nghe thử / tua trước khi tải về."""
    return name or None

def on_ref_change(new_audio, prev_state, prev_text, auto=True):
    """Khi người dùng thay/upload giọng mới, nếu giọng cũ chưa được lưu trong thư viện
    thì stash lại để cho phép lưu trước khi bị thay thế.
    Nếu bật 'Tự nhận dạng nội dung giọng mẫu' và ô nội dung đang trống thì tự trích xuất
    lời thoại trong file giọng mẫu. Trả về (prev_audio, prev_ref_text, msg, new_state, new_text)."""
    old_audio = prev_state
    unsaved_audio, unsaved_text, msg = None, "", ""
    if old_audio and new_audio != old_audio and not _voice_is_saved(old_audio):
        unsaved_audio = old_audio
        unsaved_text = prev_text or ""
        msg = ("⚠️ Giọng đang bị thay thế chưa được lưu trong thư viện. "
               "Nếu muốn giữ nó, đặt tên rồi bấm 'Lưu giọng cũ đang thay thế' — nếu không cứ để mặc, quá trình sẽ bỏ qua.")
    new_text = prev_text or ""
    if auto and new_audio and not (new_text or "").strip():
        t = transcribe_ref(new_audio)
        if t:
            new_text = t
            short = t[:80] + ("…" if len(t) > 80 else "")
            msg = (msg + "\n" if msg else "") + f"✅ Đã tự nhận dạng nội dung giọng mẫu: \"{short}\""
    return unsaved_audio, unsaved_text, msg, new_audio, new_text

def save_prev_voice_ui(name, audio_path, ref_text, model):
    """Lưu giọng cũ đang bị thay thế. Trả về (danh sách, tin nhắn, xóa stash)."""
    if not audio_path:
        return _dropdown_update(list_voices_by_model(model)), "Không có giọng cũ nào cần lưu.", None, ""
    names, msg = save_voice(name, audio_path, ref_text, model)
    return _dropdown_update(names), msg, None, ""

def save_voice_ui(name, audio_path, ref_text, model):
    names, msg = save_voice(name, audio_path, ref_text, model)
    return _dropdown_update(names), msg

def delete_voice_ui(name, model):
    _, msg = delete_voice(name)
    return _dropdown_update(list_voices_by_model(model)), msg

def _queue_choices(value):
    """Chuyển hàng chờ thành danh sách 1 dòng/task (dạng 'STT. Tóm tắt')."""
    choices = []
    for i, t in enumerate(value or [], start=1):
        text = (t.get("text") or "").replace("\n", " ")
        summary = text if len(text) <= 50 else text[:50] + "…"
        choices.append((f"{i}. {summary}", i - 1))
    return choices

def add_to_queue(queue, ref_audio, ref_text, text, instruct):
    """Thêm văn bản hiện tại vào hàng chờ, kèm theo giọng mẫu + mô tả giọng đang chọn."""
    text = (text or "").strip()
    if not text:
        raise gr.Error("Ô 'Văn bản cần đọc' đang trống — hãy nhập văn bản rồi thêm vào hàng chờ.")
    queue = list(queue or [])
    queue.append({"ref_audio": ref_audio, "ref_text": (ref_text or ""), "instruct": (instruct or "").strip() or None, "text": text})
    return queue, _dropdown_update(_queue_choices(queue)), "", -1

def clear_queue():
    """Xóa toàn bộ hàng chờ."""
    return [], _dropdown_update([]), "", -1

def store_queue_selection(queue, value):
    """Chọn task trong danh sách → lưu index và hiện ngay nội dung chi tiết."""
    idx = value if value is not None else -1
    queue = list(queue or [])
    if 0 <= idx < len(queue):
        return idx, f"Task {idx + 1}: {queue[idx]['text']}"
    return idx, ""

def view_queue_row(queue, idx):
    """Xem nội dung đầy đủ của task đang chọn."""
    queue = list(queue or [])
    if 0 <= idx < len(queue):
        return f"Task {idx + 1}: {queue[idx]['text']}"
    return "Hãy chọn 1 task trong danh sách ở trên."

def delete_queue_row(queue, idx):
    """Xóa task đang chọn khỏi hàng chờ."""
    queue = list(queue or [])
    if 0 <= idx < len(queue):
        del queue[idx]
    return queue, _dropdown_update(_queue_choices(queue)), "", -1

with gr.Blocks(title="VOICE_CLONE_TTS") as demo:
    queue_state = gr.State([])
    ref_state = gr.State()
    prev_audio_state = gr.State()
    prev_text_state = gr.State()
    selected_queue_idx = gr.State(-1)
    gr.Markdown("### Nhân bản giọng nói (OmniVoice — 600+ ngôn ngữ) — ngắt nghỉ tự nhiên, không giới hạn ký tự")
    gr.Markdown("#### Bước 1 — chọn ngôn ngữ, tải file giọng mẫu (hoặc chọn giọng đã lưu)")
    with gr.Row():
        with gr.Column():
            lang_key = gr.Dropdown(list(LANGUAGES.keys()), value=list(LANGUAGES.keys())[0], label="Ngôn ngữ kết quả (1 model duy nhất cho 600+ ngôn ngữ; chọn đúng ngôn ngữ cho chuẩn hơn)")
            saved_voices = gr.Dropdown(choices=list_voices_by_model(list(LANGUAGES.keys())[0]), label="🎤 Giọng đã lưu của ngôn ngữ này (mỗi ngôn ngữ 1 thư mục riêng) — chọn là tự nạp luôn", interactive=True)
            ref_audio = gr.Audio(type="filepath", label="v2 NGHE TRỰC TIẾP — Giọng mẫu (WAV/MP3, 5-12 giây, sạch). Để trống = dùng 'Mô tả giọng' hoặc auto voice")
            ref_text = gr.Textbox(label="Nội dung trong file giọng mẫu (để trống + bật 'Tự nhận dạng' phía dưới = tự trích xuất lời thoại từ file)", placeholder="Không bắt buộc — bật tự nhận dạng hoặc nhập chính xác thì đỡ lệch độ dài hơn")
            auto_ref_text = gr.Checkbox(value=True, label="🔊 Tự nhận dạng nội dung giọng mẫu (Whisper — lần đầu tải model nhỏ ~mất 1 phút; tự điền 'Nội dung' khi tải lên/nạp giọng mới nếu ô đang trống)")
            gen_text = gr.Textbox(label="Văn bản cần đọc (không giới hạn ký tự — tự chia đoạn ~15 giây)", lines=6)
            txt_file = gr.File(label="HOẶC tải lên file .txt chứa văn bản cần đọc (tự điền vào ô trên)", file_types=[".txt"])
            with gr.Accordion("Tùy chọn nâng cao", open=False):
                instruct = gr.Textbox(label="Mô tả giọng khi không dùng giọng mẫu (voice design) — VD: female, low pitch, british accent", placeholder="Để trống nếu dùng giọng mẫu")
                speed = gr.Slider(0.3, 2.0, value=1.0, step=0.05, label="Tốc độ đọc (speed)")
                num_step = gr.Slider(8, 64, value=16, step=1, label="Số bước khử nhiễu (num_step) — 8-16 = nhanh, tăng lên 48-64 nếu đọc thiếu chữ/rè")
                guidance = gr.Slider(1.0, 4.0, value=2.0, step=0.1, label="Cường độ bám giọng (guidance_scale)")
                chunk_duration = gr.Slider(5, 30, value=15, step=1, label="Độ dài mỗi đoạn (giây) khi văn bản dài — càng nhỏ càng tốn VRAM ít hơn")
                postprocess = gr.Checkbox(value=True, label="Loại bỏ khoảng lặng thừa + padding (postprocess_output)")
                seed_text = gr.Textbox(label="Seed (để trống = ngẫu nhiên)", placeholder="Không bắt buộc")
            queue_dd = gr.Dropdown(choices=[], label="Hàng chờ (1 dòng/task — chọn 1 task để xem hoặc xóa; nút 'Gen lần lượt hàng chờ' bên dưới sẽ gen TUẦN TỰ text hiện tại + cả hàng chờ)")
            queue_detail = gr.Textbox(label="Nội dung task đang chọn", lines=3, interactive=False)
            with gr.Row():
                btn_enqueue = gr.Button("➕ Thêm vào hàng chờ")
                btn_view_queue_row = gr.Button("👁 Xem nội dung task đang chọn")
                btn_delete_queue_row = gr.Button("🗑 Xóa task đang chọn", variant="stop")
                btn_clear_queue = gr.Button("🧹 Xóa hết")
        with gr.Column():
            out_audio = gr.Audio(type="filepath", label="Kết quả / nghe trước khi tải (task đầu tiên — các task khác xem ô bên dưới)")
            info = gr.Textbox(label="Trạng thái", interactive=False)
            queue_preview_dd = gr.Dropdown(choices=[], label="Nghe thử / tua từng task (chọn task) — có sẵn sau khi bấm Gen")
            queue_files = gr.File(label="Kết quả tất cả task (bấm tải về từng file)", file_count="multiple")
    btn_run_queue = gr.Button("▶ Gen lần lượt hàng chờ — text hiện tại + cả hàng chờ", variant="primary")
    btn_run_queue.click(gen_queue, inputs=[lang_key, ref_audio, ref_text, gen_text, instruct, speed, num_step, guidance, chunk_duration, postprocess, seed_text, queue_state], outputs=[out_audio, info, queue_files, queue_preview_dd])
    btn_enqueue.click(add_to_queue, inputs=[queue_state, ref_audio, ref_text, gen_text, instruct], outputs=[queue_state, queue_dd, queue_detail, selected_queue_idx])
    btn_clear_queue.click(clear_queue, outputs=[queue_state, queue_dd, queue_detail, selected_queue_idx])
    btn_view_queue_row.click(view_queue_row, inputs=[queue_state, selected_queue_idx], outputs=[queue_detail])
    btn_delete_queue_row.click(delete_queue_row, inputs=[queue_state, selected_queue_idx], outputs=[queue_state, queue_dd, queue_detail, selected_queue_idx])
    queue_dd.change(store_queue_selection, inputs=[queue_state, queue_dd], outputs=[selected_queue_idx, queue_detail])
    queue_preview_dd.change(load_queue_preview, inputs=[queue_preview_dd], outputs=[out_audio])

    gr.Markdown("#### Bước 2 — thư viện giọng: lưu giọng mẫu đã dùng để dùng lại lần sau (giọng sẽ được gắn theo ngôn ngữ đang chọn ở Bước 1)")
    with gr.Row():
        voice_name = gr.Textbox(label="Tên giọng mới", placeholder="VD: Giọng Nam, Giọng chị Lan...")
    with gr.Row():
        btn_save = gr.Button("💾 Lưu giọng mẫu này (theo ngôn ngữ đang chọn)")
        btn_delete = gr.Button("🗑 Xóa giọng đã chọn")
        btn_save_prev = gr.Button("💾 Lưu giọng cũ đang thay thế", variant="secondary")
    voice_msg = gr.Textbox(label="Tin nhắn thư viện giọng", interactive=False)

    btn_save.click(save_voice_ui, inputs=[voice_name, ref_audio, ref_text, lang_key], outputs=[saved_voices, voice_msg])
    saved_voices.change(load_voice_to_ui, inputs=[saved_voices, lang_key], outputs=[ref_audio, ref_text, voice_msg, ref_state, lang_key])
    btn_delete.click(delete_voice_ui, inputs=[saved_voices, lang_key], outputs=[saved_voices, voice_msg])
    btn_save_prev.click(save_prev_voice_ui, inputs=[voice_name, prev_audio_state, prev_text_state, lang_key], outputs=[saved_voices, voice_msg, prev_audio_state, prev_text_state])
    ref_audio.change(on_ref_change, inputs=[ref_audio, ref_state, ref_text, auto_ref_text], outputs=[prev_audio_state, prev_text_state, voice_msg, ref_state, ref_text])
    txt_file.change(read_txt, inputs=[txt_file], outputs=[gen_text])
    lang_key.change(refresh_voices_by_model, inputs=[lang_key, saved_voices], outputs=[saved_voices])
    demo.load(refresh_voices_by_model, inputs=[lang_key, saved_voices], outputs=[saved_voices])

try:
    demo.launch(share=True, debug=False)
except Exception as e:
    print("Không tạo được link share, chuyển sang link local:", e)
    demo.launch(share=False, debug=False)

print("Dùng xong có thể đóng tab này lại. Muốn mở lại chỉ cần chạy lại cell này.")


In [ ]:
# @title DÙNG TRỰC TIẾP BẰNG HÀM (tùy chọn)
def clone_voice(ref_file, gen_text, ref_text="", language="Vietnamese", output="/content/output/result.wav",
                speed=1.0, seed=None, num_step=16, guidance=2.0):
    """Nhân bản giọng ổn định: ref_file = file giọng mẫu, gen_text = văn bản cần đọc."""
    wave, sr, seed_used = omnivoice_generate(
        engine, gen_text, ref_audio=ref_file, ref_text=(ref_text or "").strip() or None,
        language=language or None, speed=speed, num_step=num_step, guidance=guidance, seed=seed,
    )
    path = save_wav(wave, sr, output)
    return path, sr, seed_used

# Ví dụ cách dùng (bỏ comment để chạy):
# out, sr, seed_used = clone_voice("/content/my_voice.wav", "Xin chào, đây là giọng nói được nhân bản.")
# from IPython.display import Audio
# Audio(out)


## CÁCH TẢI KẾT QUẢ VỀ MÁY
* Trong giao diện, bấm biểu tượng tải xuống (download) của ô audio kết quả.
* Hoặc vào thư mục `/content/output/` trong Files (panel bên trái) và tải xuống.

## MẸO CHẤT LƯỢNG
* File giọng mẫu nên dài **3–10 giây**, rõ ràng, không nhạc nền, không ồn.
* **Nhập chính xác "Nội dung trong file giọng mẫu"** — nếu để trống máy phải tự nhận dạng bằng Whisper (chậm hơn và có thể sai, gây lệch độ dài).
* Đọc **thiếu chữ / rè**: tăng `num_step` lên 48–64, hoặc tăng nhẹ `guidance` lên 2.5–3.0.
* Đọc **chậm**: tăng Tốc độ đọc lên 1.1–1.3.
* Với văn bản rất dài, model tự chia đoạn ~15 giây và ghép lại mượt — cứ nhập thoải mái.
* Tag cảm xúc nội dòng: `[laughter]`, `[sigh]`, `[question-en]`, `[surprise-ah]`... chèn thẳng vào văn bản.
* Nếu muốn tốc độ nhanh hơn nữa: `pip install flashinfer-python` (tùy chọn, chỉ cho GPU NVIDIA).
